# Classification and Regression with SAP HANA APL

**Automated Predictive Library (APL) — Step-by-Step Tutorial**

---

**APL (Automated Predictive Library)** is a component of SAP HANA that builds predictive models
automatically, directly inside the database. You do not need to be a data scientist to use it.
This notebook introduces APL's automated Gradient Boosting estimators for three classic supervised-learning tasks:

| Task | Class |
|---|---|
| Binary classification | `GradientBoostingBinaryClassifier` |
| Multiclass classification | `GradientBoostingClassifier` |
| Regression | `GradientBoostingRegressor` |

All three share the same core interface and the same set of advanced features — automatic variable selection, SHAP-based feature importance, and SHAP interaction values. They differ in their evaluation metrics, prediction output types, and a few class-specific methods; the multiclass and regression sections highlight exactly where they diverge.

> **Note:** APL is the same engine that powers the classification and regression
> capabilities in **SAP Analytics Cloud** (Smart Predict).

**What you will learn**

- How to connect to SAP HANA from Python using a `ConnectionContext`
- The difference between a HANA DataFrame and a pandas DataFrame
- How to train a Gradient Boosting model with a single `fit()` call
- How to evaluate the model and make predictions
- How to control the output columns with the `prediction_type` parameter
- How to generate and navigate the interactive HTML model report
- How to enable and interpret automatic variable selection
- How to compute SHAP interaction values
- How to persist a trained model to SAP HANA and reload it in a later session without retraining
- How to train one independent model per sub-population with **segmented modeling**
- Key differences between the binary, multiclass, and regression estimators

## 1. Setup

We start by importing the required libraries and configuring the APL logger.

The APL logger controls how much internal detail APL prints during training. Setting it to
`logging.WARNING` keeps the output clean — only warnings
and errors will be shown. You can change it to `logging.INFO` for a full execution trace,
or to `logging.ERROR` to suppress warnings.

In [ ]:
import logging
import pandas as pd

import hana_ml
from hana_ml import dataframe as hd
from hana_ml.algorithms.apl.gradient_boosting_classification import (
    GradientBoostingBinaryClassifier,
    GradientBoostingClassifier,
)
from hana_ml.algorithms.apl.gradient_boosting_regression import (
    GradientBoostingRegressor,
)
from hana_ml.algorithms.apl import apl_base
from hana_ml.model_storage import ModelStorage

# Keep APL log output clean — change to logging.INFO for verbose output
apl_base.config_logger(log_level=logging.WARNING)

print(f"hana_ml version: {hana_ml.__version__}")

### Connect to SAP HANA

Open a `ConnectionContext` — the object that manages the database session. Two authentication
options are available:

- **Explicit credentials** — provide host, port, user, and password directly in the notebook.
- **User key** — reference a key stored in the SAP HANA secure user store (`hdbuserstore`).
  This is the recommended approach for shared or production notebooks because credentials are
  never written in plain text.

Everything that follows runs **inside SAP HANA**. The Python client only sends instructions and
receives results; the data and the model never leave the database unless you explicitly pull
them to the client.

Two types of DataFrames are used throughout this notebook:

- **HANA DataFrame** (`hana_ml.DataFrame`) — a lazy reference to a SQL query or table in SAP HANA.
  No data is transferred to the Python client; operations on it are translated into SQL and
  executed on the database.
- **pandas DataFrame** — an in-memory table on the Python client. You obtain one by calling
  `.collect()` on a HANA DataFrame, which executes the underlying query and pulls the result
  set to the client. Use `.collect()` only when you need to inspect or plot data locally.

In [ ]:
# Option 1 — explicit credentials
HDB_HOST =
HDB_PORT =
HDB_USER =
HDB_PASS =

conn = hd.ConnectionContext(
    HDB_HOST, HDB_PORT, HDB_USER, HDB_PASS, encrypt=True, sslValidateCertificate=False
)

# Option 2 — user key stored in hdbuserstore (recommended for shared/production notebooks)
# conn = hd.ConnectionContext(userkey='mykey', encrypt=True, sslValidateCertificate=False)

print("Connected to SAP HANA.")

## 2. Binary Classification — Fraud Detection

The primary focus of this notebook is `GradientBoostingBinaryClassifier`.

We use the **Auto Claims Fraud** dataset. The goal is to predict whether an insurance claim is fraudulent (`IS_FRAUD = 'Yes'` / `'No'`).

### 2.1 Load the data

The dataset is shipped with SAP HANA APL in the `APL_SAMPLES` schema, so no file upload is needed.
We simply point a HANA DataFrame at the existing table.

In [ ]:
hdf_fraud = conn.table("AUTO_CLAIMS_FRAUD", schema="APL_SAMPLES")

print("Rows:", hdf_fraud.count())
print("Columns:", hdf_fraud.columns)
hdf_fraud.head(5).collect()

In [ ]:
FRAUD_LABEL = "IS_FRAUD"

# Class distribution
hdf_fraud.agg([("count", FRAUD_LABEL, "Count")], group_by=FRAUD_LABEL).collect()

The target column `IS_FRAUD` is binary (Yes / No). APL automatically identifies the positive class — by default it is the **least frequent** value. You can override this with the `target_key` constructor parameter.

### 2.2 Train the model

Training is one call to `fit()`. APL automatically:
- splits the data into estimation, validation, and (optionally) test partitions,
- runs gradient boosting with early stopping,
- determines the best number of iterations on the validation partition.

Use the `cutting_strategy` constructor parameter to control how the data is split across those partitions.

The only required inputs are the training `DataFrame` and the `label` column name. Specifying `key` is **strongly recommended**: it lets you join predictions back to input rows later.

**APL is a fully automated gradient boosting pipeline.** You do not need to tune hyperparameters — APL selects optimal values internally and is designed to produce excellent results quickly, making it a cost-effective AutoML solution. Although parameters such as `learning_rate`, `max_depth`, and `max_iterations` are exposed in the constructor, users are not expected to set them. They are provided only for advanced users who need fine-grained control.

In [ ]:
FRAUD_KEY = "CLAIM_ID"

model_bc = GradientBoostingBinaryClassifier()

model_bc.fit(
    data=hdf_fraud,
    key=FRAUD_KEY,
    label=FRAUD_LABEL,
)

### 2.3 Make predictions

In [ ]:
predictions = model_bc.predict(hdf_fraud)
predictions.head(10).collect()

The default `predict()` output contains four columns:

| Column | Content |
|---|---|
| `CLAIM_ID` | The key — useful for joining back to source data |
| `TRUE_LABEL` | Actual label (returned when the input contains the label column) |
| `PREDICTED` | Predicted label (`Yes` / `No`) |
| `PROBABILITY` | Probability associated with the predicted label |

### 2.4 Human-readable explanations

Getting a predicted label and a probability is often not enough to take action.
To prevent a fraudulent claim from being paid, you need to know *what* drove the high fraud probability for that specific claim — not just that it is high.
This is the domain of **local interpretability**: explaining an individual prediction in terms of the input variables that influenced it.

Setting `prediction_type='Explanations'` produces a **long-format** table where each row
describes the contribution of one influencer to one record's prediction.
An **influencer** is an input variable that was selected and used by the model during training —
variables that were not selected do not appear in this output.

In [ ]:
expl_df = model_bc.predict(hdf_fraud, prediction_type="Explanations")
expl_df.collect().sort_values([FRAUD_KEY, "Explanation_Rank"]).head(20)

The output columns are:

| Column | Content |
|---|---|
| `CLAIM_ID` | The key — one or more rows per input record |
| `TRUE_LABEL` | Actual label (when the input contains the label column) |
| `PREDICTED` | Predicted class label |
| `Explanation_Rank` | Rank of this influencer among all influencers for the record (1 = strongest contributor) |
| `Explanation_Influencer` | Name of the influencer (input variable selected by the model) |
| `Explanation_Influencer_Value` | Actual value of that variable for this record |
| `Explanation_Strength` | Signed strength score (see below) |

#### The strength score

The strength is a **normalized** measure of the influencer's impact on the prediction. It is derived from the SHAP value of the influencer divided by the standard deviation of all SHAP contributions, so it can be interpreted as a distance from the average expressed in number of standard deviations.

- A **positive** strength means the variable pushed the prediction toward the positive class (or toward a higher value for regression).
- A **negative** strength means the variable pushed it away.

The strength groups into six qualitative buckets (thresholds at ±1 and ±3 correspond to the usual 1σ / 3σ distances from the mean in a normal distribution):

| Range | Group |
|---|---|
| > 3 | Strong positive |
| > 1 | Meaningful positive |
| > 0 | Weak positive |
| ≥ −1 | Weak negative |
| ≥ −3 | Meaningful negative |
| < −3 | Strong negative |

If you need the **raw, unnormalized SHAP values** — one numeric column per feature — use `prediction_type='Individual Contributions'` instead.

#### Positive Others and Negative Others

APL limits the output to at most **10 explanations per record**. When the model uses more than 10 influencers, the remaining ones are aggregated into two synthetic entries: `Positive Others` (sum of all small positive contributions) and `Negative Others` (sum of all small negative contributions). Their combined strength can be significant even though each individual contribution was small.

### 2.5 Interactive HTML report

The interactive HTML report lets you explore the model without writing any code.
It is the fastest way to assess overall quality, understand which variables the model relies on,
and inspect individual predictions visually — useful for sharing results with stakeholders or
reviewing a model before deployment.

Call `build_report()` to compile the report data. Running `predict()` with
`prediction_type='Explanations'` first means the explanation data is available when
`build_report()` runs, so the Local Explanations section is populated with per-row charts.
Then use `generate_notebook_iframe_report()` to embed the report directly in the notebook,
or `generate_html_report()` to save it as a standalone HTML file you can share.

The report is organized into seven sections:

| Section | Content |
|---|---|
| **Statistic** | Key quality metrics on the validation partition: AUC, Balanced Classification Rate, Predictive Power (Gini coefficient, Gini = 2 × AUC − 1), and more |
| **Parameter** | Parameters used for training |
| **Optimal Parameter** | The best boosting iteration found by early stopping |
| **Confusion Matrix** | Actual vs. predicted class counts on the validation partition |
| **Variable Importance** | Bar chart of each variable's relative contribution to the model |
| **Metrics** | ROC curve, Cumulative Lift curve, and Cumulative Gains curve |
| **Local Explanations** | Per-row stacked bar charts showing the normalized strength of each influencer — color-coded by contribution category (strong/meaningful/weak, positive/negative) |

> **Shared report structure.** This same report layout is used by all hana-ml classification
> and regression estimators — both APL's gradient boosting models and SAP HANA PAL's
> supervised learning algorithms.

In [ ]:
model_bc.build_report()
model_bc.generate_notebook_iframe_report()

# Alternatively, save to a shareable HTML file
model_bc.generate_html_report("fraud_model_report.html")

### 2.6 Programmatic access to model outputs

The interactive report summarizes the model visually. This section shows how to access the
same information programmatically — useful when you need to extract specific numbers, log
results, or feed them into downstream systems.

Some information is available through direct functions on the model object; other details
are available through the debrief reports that APL can generate from the trained model.

#### Performance metrics

`get_performance_metrics()` returns a dict of metrics measured on the **validation** partition.

In [ ]:
KEY_METRICS = ["AUC", "PredictivePower", "BalancedClassificationRate"]

metrics = model_bc.get_performance_metrics()
pd.DataFrame({"Value": {k: metrics[k] for k in KEY_METRICS}})

#### Feature importance

In [ ]:
importances = model_bc.get_feature_importances()
pd.DataFrame(importances).sort_values("ExactSHAP", ascending=False)

#### Detailed debrief reports

`get_debrief_report()` gives programmatic access to the individual statistical reports that APL
can generate from the trained model. Each report returns a HANA DataFrame that you can collect
and analyze.

The full list of available reports and their contents is documented in the
[APL Classification and Regression Debrief Reports reference](https://help.sap.com/docs/apl/7223667230cb471ea916200712a9c682/1b5b9d7d317b42b3bbed3aa32932a3c5.html).

In [ ]:
# Global performance metrics — one row per indicator per partition
perf = model_bc.get_debrief_report("ClassificationRegression_Performance")
perf.deselect("Oid").filter("\"Partition\" = 'Validation'").collect()

In [ ]:
# Variable contribution
var_contrib = model_bc.get_debrief_report(
    "ClassificationRegression_VariablesContribution"
)
var_contrib.deselect("Oid").collect().sort_values("Contribution", ascending=False)

In [ ]:
# Confusion matrix
cm = model_bc.get_debrief_report("Classification_BinaryClass_ConfusionMatrix")
cm.deselect("Oid").filter("\"Partition\" = 'Validation'").collect()

In [ ]:
# ROC curve data
roc = model_bc.get_debrief_report("BinaryTarget_CurveRoc")
roc.deselect("Oid").head(10).collect()

### 2.7 Automatic variable selection

APL's gradient boosting performs implicit variable selection: variables that do not contribute to the model are naturally assigned zero importance and have no effect on predictions. Enabling `variable_auto_selection=True` goes further — it explicitly removes low-importance variables through a multi-step process that:

1. Trains a reference model with all variables.
2. Iteratively removes the least important variables.
3. Stops when removing more variables would degrade model quality by more than `variable_selection_quality_bar` (default: 0.01 AUC).

The result is a strictly smaller set of variables, retaining almost the same predictive power while making the model simpler and easier to audit.

**Why does this matter?**  
Fewer features reduce overfitting risk, cut prediction latency, and make the model easier to explain to stakeholders.

In [ ]:
model_bc_sel = GradientBoostingBinaryClassifier(
    variable_auto_selection=True,
    # Allow at most 0.02 AUC loss compared to the full-feature reference model.
    # Increasing this bar accepts more quality loss and yields fewer variables.
    variable_selection_quality_bar=0.02,
    # Hard cap: keep at most 8 variables regardless of quality bar.
    variable_selection_max_nb_of_final_variables=8,
)

model_bc_sel.fit(
    data=hdf_fraud,
    key=FRAUD_KEY,
    label=FRAUD_LABEL,
)

#### Inspect which variables were selected

APL exposes four debrief reports that trace the variable selection process step by step:

| Report | Content |
|---|---|
| `ClassificationRegression_VariablesSelectionSummary` | Number of input variables at each step; flags which step produced the selected model |
| `ClassificationRegression_VariablesSelectionDetails` | Variables excluded at each step |
| `ClassificationRegression_VariablesSelectionPerformance` | Model quality at each step |
| `ClassificationRegression_VariablesExclusion` | All excluded variables together with the reason for exclusion |

In [ ]:
# Step-by-step summary: how many variables remain at each step, and which step was selected
summary = model_bc_sel.get_debrief_report(
    "ClassificationRegression_VariablesSelectionSummary"
)
summary.deselect("Oid").collect()

In [ ]:
# Which variables were dropped at each step
details = model_bc_sel.get_debrief_report(
    "ClassificationRegression_VariablesSelectionDetails"
)
details.deselect("Oid").collect()

In [ ]:
# Model quality at each step — shows the trade-off between fewer variables and predictive power
perf_sel = model_bc_sel.get_debrief_report(
    "ClassificationRegression_VariablesSelectionPerformance"
)
perf_sel.deselect("Oid").collect()

The selected model uses 7 variables instead of 12 for virtually the same AUC — a smaller, simpler model that is cheaper to deploy and easier to explain.

In [ ]:
# All excluded variables with the reason for their exclusion
excl = model_bc_sel.get_debrief_report("ClassificationRegression_VariablesExclusion")
excl.deselect("Oid").collect()

### 2.8 SHAP interaction values

SHAP interaction values quantify how much **pairs** of features jointly influence predictions, beyond their individual contributions. For example, the combination *AGE=young* and *GENDER=male* might have a larger combined effect than the sum of their individual SHAP values.

SHAP interactions are **disabled by default** because they require additional computation during training. Enable them with `interactions=True`.

In [ ]:
model_bc_inter = GradientBoostingBinaryClassifier(
    interactions=True,
    # Keep at most the top 5 interactions per variable (default)
    interactions_max_kept=5,
)

model_bc_inter.fit(
    data=hdf_fraud,
    key=FRAUD_KEY,
    label=FRAUD_LABEL,
    build_report=True,
)

The Interaction Matrix tab is now populated in the report. Each cell of the interaction matrix represents a (variable, interacting variable) pair, with the average absolute interaction SHAP value.

In [ ]:
model_bc_inter.generate_notebook_iframe_report()

The interaction matrix is also available through `get_debrief_report('ClassificationRegression_InteractionMatrix')`.  

In [ ]:
interactions_df = model_bc_inter.get_debrief_report(
    "ClassificationRegression_InteractionMatrix"
)
interactions_df.deselect("Oid").head(20).collect()

> **Note on interaction index methodology.** APL's SHAP interaction values are computed using the **Shapley-Taylor interaction index**, which provides a more principled quantification of feature interactions than the method used by popular gradient boosting libraries such as XGBoost, LightGBM, or CatBoost. The Shapley-Taylor index decomposes the model's output into a sum of individual and pairwise effects in a way that satisfies stronger axiomatic properties. Expert users interested in the theoretical foundations may refer to the original paper: [Shapley-Taylor Interaction Index](https://arxiv.org/pdf/1902.05622).

### 2.9 Saving and reloading the model

Use `ModelStorage` to persist a trained model in SAP HANA so it can be reloaded in a future session without retraining.

In [ ]:
from hana_ml.model_storage import ModelStorage

FRAUD_MODEL_NAME = "Fraud Detection — Binary Classifier"

# Choose a schema that the current user can write to
model_storage = ModelStorage(connection_context=conn, schema="USER_APL")

model_bc.name = FRAUD_MODEL_NAME

# Save — if_exists='replace' overwrites any existing model with the same name and version
model_storage.save_model(model=model_bc, if_exists="replace")

# List all stored models
model_storage.list_models(name=FRAUD_MODEL_NAME)

In [ ]:
# Reload the model from the registry — simulates loading in a separate session
loaded_model = model_storage.load_model(name=FRAUD_MODEL_NAME)

# Generate new predictions with the reloaded model — no retraining needed
predictions_loaded = loaded_model.predict(hdf_fraud)

print("Predictions from the reloaded model:")
predictions_loaded.head(5).collect()

In [ ]:
# Clean up: remove the saved model from the registry
model_storage.delete_model(name=FRAUD_MODEL_NAME, version=1)
print(f'Model "{FRAUD_MODEL_NAME}" removed from storage.')

# Verify it is gone
remaining = model_storage.list_models(name=FRAUD_MODEL_NAME)
if remaining.empty:
    print("No models found — cleanup complete.")

### 2.10 Segmented Modeling

So far every model in this notebook was trained on the full dataset as a single unit.
APL's gradient boosting estimators also support **segmented modeling**: training one independent
model per unique value of a segment column in a single call.

**Why segmented modeling?**

- Different sub-populations can exhibit very different patterns. A single pooled model averages
  those differences away.
- Segmented modeling lets APL select the best parameters **independently** for each group, then
  return all predictions in one combined table.
- The API is identical to the single-model workflow — the only addition is the
  `segment_column_name` constructor argument.

**Dataset:** the same Auto Claims Fraud dataset used throughout this section.  
**Segment column:** `INCOME_CATEGORY` — integer income brackets (e.g. 14, 25, 35, 50).
Fraud patterns may differ across income groups.

In [ ]:
SEGMENT_COLUMN = "INCOME_CATEGORY"

hdf_fraud = hdf_fraud.sort(SEGMENT_COLUMN)

# Inspect the segment distribution before training
hdf_fraud.agg([("count", SEGMENT_COLUMN, "Count")], group_by=SEGMENT_COLUMN).sort(
    SEGMENT_COLUMN
).collect()

#### Train the segmented model

Setting `segment_column_name` tells APL to train **one independent model per unique value
of that column**. All models are built in a single `fit()` call; APL handles the parallelism
internally.

`max_tasks` controls how many models are trained in parallel. The default is `1` (sequential).
Setting it to `0` tells APL to use **all available HANA threads**, which can significantly
reduce wall-clock time when training many segments.

The predictions and all programmatic outputs include the segment column, so you can filter
results down to a specific group.

To show how APL handles partial failures, segment `14` is intentionally given only 3 rows —
too few for APL, which fails for that segment. The other segments
train normally.

In [ ]:
# Simulate a data pipeline issue for one segment: keep only 3 rows for INCOME_CATEGORY=14.
# APL cannot build a model with so few data points; the other segments train normally.
hdf_few = hdf_fraud.filter(f'"{SEGMENT_COLUMN}" = 14').head(3)
hdf_other = hdf_fraud.filter(f'"{SEGMENT_COLUMN}" != 14')
hdf_reduced = hdf_other.union(hdf_few).sort(SEGMENT_COLUMN)

model_bc_seg = GradientBoostingBinaryClassifier(
    segment_column_name=SEGMENT_COLUMN,
    max_tasks=0,  # use all available HANA threads for parallel segment training
    target_key="Yes",  # mandatory for segmented models
)

model_bc_seg.fit(
    data=hdf_reduced,
    key=FRAUD_KEY,
    label=FRAUD_LABEL,
)

print("Training complete.")

#### Predictions

The prediction output is the same as for a single-model call.
Filter on `INCOME_CATEGORY` to inspect results for a specific segment.

In [ ]:
predictions_seg = model_bc_seg.predict(hdf_reduced)

# Preview predictions for one income group
predictions_seg.filter(f'"{SEGMENT_COLUMN}" = 25').head(5).collect()

#### Per-segment performance metrics

`get_performance_metrics()` returns a DataFrame that includes the segment column when the
model was trained in segmented mode, making it easy to compare model quality across groups.

All other model outputs also work in segmented mode and include the segment column:
`get_feature_importances()`, `get_debrief_report()`, etc.

In [ ]:
metrics_seg = model_bc_seg.get_performance_metrics()
metrics_seg[metrics_seg["Metric"] == "AUC"].sort_values(SEGMENT_COLUMN)

### Troubleshooting

Training may succeed for some segments while failing for others. The overall `fit()` call
does **not** raise an exception in that case — the successful segments still produce models.

**Automatic warning**

When at least one segment fails, APL automatically emits a `WARNING` log message listing the
first 10 failed segments and the corresponding error. You can see this in the `fit()`
output above. For a small number of failures this is usually sufficient to diagnose the problem.

**Checking task status per segment**

When there are more than 10 failures, the warning does not contain the full list. Use
`get_summary()` to retrieve all failed segments programmatically.
Filter on `AplTaskStatus` to get a quick overview of which segments succeeded and which failed:

In [ ]:
df_status = (
    model_bc_seg.get_summary()
    .filter("\"KEY\" in ('AplTaskStatus')")
    .select("OID", "VALUE")
    .collect()
)
df_status.columns = [SEGMENT_COLUMN, "Task Status"]
df_status

**Inspecting the failure log for a specific segment**

For any segment that failed, `get_fit_operation_log()` gives the full APL log messages for
that segment.
Filter to `LEVEL = 0` (top-level messages) and the segment's `OID` to surface the root cause:

In [ ]:
df_log = (
    model_bc_seg.get_fit_operation_log()
    .filter("LEVEL = 0 and OID = '14'")
    .select("OID", "MESSAGE")
    .collect()
)
df_log.columns = [SEGMENT_COLUMN, "Log Text"]
df_log

## 3. Multiclass Classification

This section demonstrates `GradientBoostingClassifier` on a multiclass target. The API is identical to `GradientBoostingBinaryClassifier`; this section covers only what is different.

We use the **Census** dataset and predict `marital-status` (7 classes).

In [ ]:
CENSUS_LABEL_MC = "marital-status"

hdf_census = conn.table("CENSUS", schema="APL_SAMPLES")

# Show the class distribution for the target
hdf_census.agg(
    [("count", CENSUS_LABEL_MC, "Count")], group_by=CENSUS_LABEL_MC
).collect()

### 3.1 Train and predict

The key difference versus binary classification:

- The default `eval_metric` is `'MultiClassLogLoss'` (binary uses `'LogLoss'`).
- There is **no** `target_key` parameter — the positive class concept does not apply.
- For variable selection, the quality bar is expressed as a maximum BCR (Balanced Classification Rate) loss, not AUC loss.

In [ ]:
CENSUS_KEY = "id"

model_mc = GradientBoostingClassifier(
    variable_auto_selection=True,
    variable_selection_max_nb_of_final_variables=6,
)

model_mc.fit(
    data=hdf_census,
    key=CENSUS_KEY,
    label=CENSUS_LABEL_MC,
    build_report=True,
)

In [ ]:
model_mc.get_performance_metrics()

### 3.2 Per-class metrics

`get_metrics_per_class()` is available only on `GradientBoostingClassifier`. It returns precision, recall, and F1 score broken down by class — essential for diagnosing which classes are well-predicted and which are not.

In [ ]:
per_class = model_mc.get_metrics_per_class()

pd.DataFrame(per_class).sort_values("F1Score", ascending=False)

### 3.3 Prediction outputs for multiclass

The default output is identical to binary classification: `PREDICTED` and `PROBABILITY`. In addition, you can request **probabilities for all classes** with `prediction_type='AllProbabilities'`.

In [ ]:
# Default prediction — one column per row for the predicted class and its probability
pred_mc = model_mc.predict(hdf_census)
pred_mc.head(5).collect()

In [ ]:
# All class probabilities — one column per class (PROBA_<class_value>)
pred_mc_all = model_mc.predict(hdf_census, prediction_type="AllProbabilities")
pred_mc_all.head(5).collect()

### 3.4 Confusion matrix debrief report

`GradientBoostingClassifier` provides an additional debrief report not available for binary targets.

In [ ]:
confusion = model_mc.get_debrief_report("Classification_MultiClass_ConfusionMatrix")
confusion.filter("\"Partition\" = 'Validation'").deselect("Oid").collect()

## 4. Regression

This section demonstrates `GradientBoostingRegressor` on a continuous target. We predict `age` from the Census dataset.

Only the differences relative to binary classification are highlighted.

### 4.1 Train the model

Differences versus the classifiers:

- The default `eval_metric` is `'RMSE'`.
- For variable selection, the quality bar is expressed as a maximum relative RMSE increase.

In [ ]:
model_reg = GradientBoostingRegressor(
    variable_auto_selection=True,
)

model_reg.fit(
    data=hdf_census,
    key=CENSUS_KEY,
    label="age",
)

In [ ]:
metrics_reg = model_reg.get_performance_metrics()
pd.DataFrame({"Value": {k: metrics_reg[k] for k in ["R2", "RootMeanSquareError"]}})

### 4.2 Predictions and explanations

In [ ]:
pred_reg = model_reg.predict(hdf_census)
pred_reg.head(5).collect()

In [ ]:
expl_reg = model_reg.predict(hdf_census, prediction_type="Explanations")
expl_reg.head(5).collect()

The **Local Explanations** tab of the HTML report visualizes the same data as a **waterfall chart** — one chart per record (up to 100 by default; adjust with `max_local_explanations`).

Each chart is a horizontal bar chart where the bars form a running total of SHAP contributions:

- **Y-axis** — the influencers, labeled as `variable_name = variable_value` (e.g. `marital-status = Married`). They are ordered from strongest contributor at the top to weakest at the bottom.
- **X-axis** — the contribution value (`Explanation_Contribution`): the raw, unnormalized SHAP value in the same unit as the target.
- **Baseline bar** — the first bar, representing the model's average prediction across the training set. It is the starting reference point for every prediction.
- **Influencer bars** — each bar starts where the previous one ended (a running cumulative total), then extends by that influencer's contribution:
  - **Green** → positive contribution: this feature value pushed the prediction *up* relative to the baseline.
  - **Red** → negative contribution: this feature value pushed the prediction *down*.
- **Predicted Value bar** — a single dark-gray summary bar spanning from zero to the final cumulative total, making the net prediction offset immediately visible.

To reconstruct the prediction from the chart: start at the Baseline, add every influencer bar in order, and the running total at the end equals the model's predicted value for that record.

In [ ]:
model_reg.build_report()
model_reg.generate_notebook_iframe_report()